# Model Diffing: Tracking Representational Changes

**Survey Reference:** Section 12.1 - Model Diffing: Tracking Representational Changes Across Models

## Overview

**Model diffing** compares representations between different models or model versions to understand:
- What changed after fine-tuning?
- How do different architectures represent similar concepts?
- Which model components are most different?

## Learning Objectives

1. Compare representations using CKA and other metrics
2. Identify where models differ most
3. Track changes from fine-tuning
4. Visualize representational differences

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Representation Similarity Metrics

### Centered Kernel Alignment (CKA)

In [ ]:
def center(X: torch.Tensor) -> torch.Tensor:
    """Center columns of X."""
    return X - X.mean(dim=0, keepdim=True)

def linear_CKA(X: torch.Tensor, Y: torch.Tensor) -> float:
    """
    Compute linear CKA between two representation matrices.
    
    Args:
        X: [n_samples, dim_x]
        Y: [n_samples, dim_y]
    """
    X = center(X)
    Y = center(Y)
    
    XTX = X.T @ X
    YTY = Y.T @ Y
    YTX = Y.T @ X
    
    cka = (YTX.norm() ** 2) / (XTX.norm() * YTY.norm() + 1e-8)
    return cka.item()

def rbf_CKA(X: torch.Tensor, Y: torch.Tensor, sigma: float = None) -> float:
    """
    Compute RBF kernel CKA (captures nonlinear relationships).
    """
    def rbf_kernel(Z, sigma):
        dists = torch.cdist(Z, Z)
        return torch.exp(-dists**2 / (2 * sigma**2))
    
    if sigma is None:
        sigma = X.shape[1] ** 0.5  # Default: sqrt of dimension
    
    K_X = rbf_kernel(X, sigma)
    K_Y = rbf_kernel(Y, sigma)
    
    # Center kernels
    n = K_X.shape[0]
    H = torch.eye(n) - torch.ones(n, n) / n
    K_X = H @ K_X @ H
    K_Y = H @ K_Y @ H
    
    cka = (K_X * K_Y).sum() / (K_X.norm() * K_Y.norm() + 1e-8)
    return cka.item()

### Other Metrics

In [ ]:
def cosine_similarity_mean(X: torch.Tensor, Y: torch.Tensor) -> float:
    """Average cosine similarity between corresponding vectors."""
    # Assumes X and Y have same shape [n_samples, dim]
    sims = F.cosine_similarity(X, Y, dim=1)
    return sims.mean().item()

def procrustes_distance(X: torch.Tensor, Y: torch.Tensor) -> float:
    """
    Procrustes distance: minimum distance after optimal rotation.
    """
    # Normalize
    X = X / X.norm()
    Y = Y / Y.norm()
    
    # SVD of Y^T X to find optimal rotation
    M = Y.T @ X
    U, S, Vh = torch.linalg.svd(M)
    R = U @ Vh  # Optimal rotation
    
    # Distance after rotation
    Y_rotated = Y @ R
    return (X - Y_rotated).norm().item()

## 2. Comparing Two Models

In [ ]:
def collect_representations(model, tokenizer, texts, layer_idx):
    """
    Collect representations from a model for comparison.
    """
    reps = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        # Mean pool over sequence
        h = outputs.hidden_states[layer_idx].mean(dim=1)  # [1, hidden_size]
        reps.append(h)
    
    return torch.cat(reps, dim=0)  # [n_texts, hidden_size]

# Load two models to compare
model1_name = "gpt2"
model2_name = "gpt2-medium"

tokenizer1 = AutoTokenizer.from_pretrained(model1_name)
model1 = AutoModelForCausalLM.from_pretrained(model1_name).to(device).eval()

tokenizer2 = AutoTokenizer.from_pretrained(model2_name)
model2 = AutoModelForCausalLM.from_pretrained(model2_name).to(device).eval()

In [ ]:
# Sample texts for comparison
comparison_texts = [
    "The capital of France is Paris.",
    "Machine learning is a subset of artificial intelligence.",
    "The quick brown fox jumps over the lazy dog.",
    "In 1969, humans first landed on the moon.",
    "Python is a popular programming language.",
]

## 3. Layer-wise Comparison

In [ ]:
def compare_models_layerwise(model1, model2, tokenizer1, tokenizer2, texts):
    """
    Compare two models at each layer using CKA.
    """
    n_layers1 = model1.config.n_layer + 1
    n_layers2 = model2.config.n_layer + 1
    
    similarity_matrix = np.zeros((n_layers1, n_layers2))
    
    for l1 in range(n_layers1):
        reps1 = collect_representations(model1, tokenizer1, texts, l1)
        for l2 in range(n_layers2):
            reps2 = collect_representations(model2, tokenizer2, texts, l2)
            
            # Use CKA (works with different dimensions)
            cka = linear_CKA(reps1, reps2)
            similarity_matrix[l1, l2] = cka
    
    return similarity_matrix

# This takes a while - uncomment to run
# sim_matrix = compare_models_layerwise(model1, model2, tokenizer1, tokenizer2, comparison_texts)

In [ ]:
def plot_similarity_matrix(sim_matrix, title="Layer-wise CKA"):
    """
    Visualize layer-wise similarity.
    """
    plt.figure(figsize=(10, 8))
    plt.imshow(sim_matrix, cmap='viridis', aspect='auto')
    plt.colorbar(label='CKA')
    plt.xlabel('Model 2 Layer')
    plt.ylabel('Model 1 Layer')
    plt.title(title)
    plt.show()

# plot_similarity_matrix(sim_matrix, f"CKA: {model1_name} vs {model2_name}")

## 4. Tracking Fine-tuning Changes

In [ ]:
def compare_before_after_finetuning(base_model, finetuned_model, tokenizer, texts):
    """
    Compare a base model to its fine-tuned version.
    """
    n_layers = base_model.config.n_layer + 1
    
    layer_changes = []
    
    for layer_idx in range(n_layers):
        base_reps = collect_representations(base_model, tokenizer, texts, layer_idx)
        ft_reps = collect_representations(finetuned_model, tokenizer, texts, layer_idx)
        
        # Measure change (1 - CKA = amount of change)
        cka = linear_CKA(base_reps, ft_reps)
        change = 1 - cka
        layer_changes.append(change)
    
    return layer_changes

# TODO: Compare base and fine-tuned versions of same model

## 5. Weight Space Comparisons

Beyond activations, we can also compare weights directly.

In [ ]:
def compare_weights(model1, model2):
    """
    Compare model weights directly.
    """
    weight_diffs = {}
    
    state1 = model1.state_dict()
    state2 = model2.state_dict()
    
    for name in state1:
        if name in state2 and state1[name].shape == state2[name].shape:
            w1 = state1[name].float()
            w2 = state2[name].float()
            
            # Relative change
            diff = (w1 - w2).norm() / (w1.norm() + 1e-8)
            weight_diffs[name] = diff.item()
    
    return weight_diffs

# For same-architecture models (e.g., base vs fine-tuned)
# diffs = compare_weights(base_model, finetuned_model)

## 6. Semantic Similarity Analysis

Compare how models represent specific concepts.

In [ ]:
def compare_concept_representations(model1, model2, tokenizer1, tokenizer2, 
                                    concept_examples: dict, layer_idx: int):
    """
    Compare how models represent different concepts.
    
    Args:
        concept_examples: Dict mapping concept names to example texts
    """
    results = {}
    
    for concept, examples in concept_examples.items():
        reps1 = collect_representations(model1, tokenizer1, examples, layer_idx)
        reps2 = collect_representations(model2, tokenizer2, examples, layer_idx)
        
        cka = linear_CKA(reps1, reps2)
        results[concept] = cka
    
    return results

# Example concepts
concept_examples = {
    "science": ["Physics explains how the universe works.", "Chemistry studies matter and its interactions."],
    "animals": ["Dogs are loyal companions.", "Cats are independent pets."],
    "geography": ["Paris is the capital of France.", "Tokyo is the capital of Japan."],
}

# concept_sims = compare_concept_representations(model1, model2, tokenizer1, tokenizer2, 
#                                                 concept_examples, layer_idx=6)

## Key Takeaways

1. **CKA** is the standard metric for comparing representations across models
2. **Layer-wise comparison** reveals how representations align
3. **Fine-tuning changes** can be localized to specific layers
4. **Weight comparisons** complement activation analysis
5. **Concept-specific analysis** shows which representations transfer

## Next Steps

→ **02_knowledge_localization.ipynb**: Find where specific knowledge is stored